<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">처음부터 만드는 대형 언어 모델</a> 책의 보조 코드 by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 3장: 어텐션 메커니즘 코딩

이 노트북에서 사용되는 패키지들:

In [ ]:
from importlib.metadata import version

print("torch version:", version("torch"))

- 이 장에서는 LLM의 엔진인 어텐션 메커니즘을 다룹니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/01.webp?123" width="500px">

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/02.webp" width="600px">

## 3.1 긴 시퀀스 모델링의 문제점

- 이 섹션에는 코드가 없습니다
- 소스 언어와 타겟 언어 간의 문법 구조 차이로 인해 텍스트를 단어별로 번역하는 것은 불가능합니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/03.webp" width="400px">

- 트랜스포머 모델이 도입되기 전에는 인코더-디코더 RNN이 기계 번역 작업에 일반적으로 사용되었습니다
- 이 설정에서 인코더는 소스 언어의 토큰 시퀀스를 처리하여 숨겨진 상태(신경망 내의 중간 레이어의 일종)를 사용하여 전체 입력 시퀀스의 압축된 표현을 생성합니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/04.webp" width="500px">

## 3.2 어텐션 메커니즘으로 데이터 의존성 포착하기

- 이 섹션에는 코드가 없습니다
- 어텐션 메커니즘을 통해 네트워크의 텍스트 생성 디코더 부분이 모든 입력 토큰에 선택적으로 접근할 수 있으며, 이는 특정 출력 토큰 생성에 있어 일부 입력 토큰이 다른 토큰보다 더 중요하다는 것을 의미합니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/05.webp" width="500px">

- 트랜스포머의 셀프 어텐션(self-attention)은 시퀀스의 각 위치가 같은 시퀀스 내의 다른 모든 위치와 관계를 맺고 관련성을 결정할 수 있도록 하여 입력 표현을 향상시키는 기술입니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/06.webp" width="300px">

## 3.3 셀프 어텐션으로 입력의 다른 부분에 주의 기울이기

### 3.3.1 훈련 가능한 가중치가 없는 간단한 셀프 어텐션 메커니즘

- 이 섹션에서는 훈련 가능한 가중치가 포함되지 않은 매우 단순화된 셀프 어텐션 변형을 설명합니다
- 이는 순전히 설명 목적이며 트랜스포머에서 사용되는 어텐션 메커니즘이 아닙니다
- 다음 섹션인 3.3.2에서는 이 간단한 어텐션 메커니즘을 확장하여 실제 셀프 어텐션 메커니즘을 구현할 것입니다
- 입력 시퀀스 $x^{(1)}$부터 $x^{(T)}$까지가 주어졌다고 가정합니다
  - 입력은 2장에서 설명한 대로 토큰 임베딩으로 변환된 텍스트(예: "Your journey starts with one step"과 같은 문장)입니다
  - 예를 들어, $x^{(1)}$은 "Your"라는 단어를 나타내는 d차원 벡터입니다
- **목표:** $x^{(1)}$부터 $x^{(T)}$까지의 각 입력 시퀀스 요소 $x^{(i)}$에 대해 컨텍스트 벡터 $z^{(i)}$를 계산합니다 ($z$와 $x$는 동일한 차원을 가집니다)
    - 컨텍스트 벡터 $z^{(i)}$는 입력 $x^{(1)}$부터 $x^{(T)}$까지의 가중합입니다
    - 컨텍스트 벡터는 특정 입력에 "컨텍스트" 특화적입니다
      - 임의의 입력 토큰에 대한 자리표시자인 $x^{(i)}$ 대신에, 두 번째 입력인 $x^{(2)}$를 고려해보겠습니다
      - 구체적인 예를 계속하기 위해, 자리표시자 $z^{(i)}$ 대신에 두 번째 출력 컨텍스트 벡터인 $z^{(2)}$를 고려합니다
      - 두 번째 컨텍스트 벡터 $z^{(2)}$는 두 번째 입력 요소 $x^{(2)}$에 대해 가중된 모든 입력 $x^{(1)}$부터 $x^{(T)}$까지의 가중합입니다
      - 어텐션 가중치는 $z^{(2)}$를 계산할 때 가중합에서 각 입력 요소가 얼마나 기여하는지를 결정하는 가중치입니다
      - 간단히 말해서, $z^{(2)}$는 주어진 작업에 관련된 다른 모든 입력 요소에 대한 정보도 통합하는 $x^{(2)}$의 수정된 버전이라고 생각하면 됩니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/07.webp" width="400px">

- (이 그림의 숫자들은 시각적 혼란을 줄이기 위해 소수점 이하 한 자리로 잘렸습니다; 마찬가지로 다른 그림들도 잘린 값들을 포함할 수 있습니다)

- 관례에 따라, 정규화되지 않은 어텐션 가중치는 **"어텐션 점수(attention scores)"**라고 하며, 합이 1이 되도록 정규화된 어텐션 점수는 **"어텐션 가중치(attention weights)"**라고 합니다

- 아래 코드는 위 그림을 단계별로 설명합니다

<br>

- **1단계:** 정규화되지 않은 어텐션 점수 $\omega$ 계산
- 두 번째 입력 토큰을 쿼리로 사용한다고 가정하면, 즉 $q^{(2)} = x^{(2)}$일 때, 내적을 통해 정규화되지 않은 어텐션 점수를 계산합니다:
    - $\omega_{21} = x^{(1)} q^{(2)\top}$
    - $\omega_{22} = x^{(2)} q^{(2)\top}$
    - $\omega_{23} = x^{(3)} q^{(2)\top}$
    - ...
    - $\omega_{2T} = x^{(T)} q^{(2)\top}$
- 위에서 $\omega$는 정규화되지 않은 어텐션 점수를 나타내는 데 사용되는 그리스 문자 "오메가"입니다
    - $\omega_{21}$의 아래첨자 "21"은 입력 시퀀스 요소 2가 입력 시퀀스 요소 1에 대한 쿼리로 사용되었음을 의미합니다

- 3장에서 설명한 대로 이미 3차원 벡터로 임베딩된 다음 입력 문장이 있다고 가정해보겠습니다 (여기서는 줄바꿈 없이 페이지에 맞도록 설명 목적으로 매우 작은 임베딩 차원을 사용합니다):

In [ ]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

- (이 책에서는 훈련 예시가 행으로, 특성값이 열로 표현되는 일반적인 기계학습 및 딥러닝 관례를 따릅니다; 위에 표시된 텐서의 경우 각 행은 단어를, 각 열은 임베딩 차원을 나타냅니다)

- 이 섹션의 주요 목표는 두 번째 입력 시퀀스 $x^{(2)}$를 쿼리로 사용하여 컨텍스트 벡터 $z^{(2)}$가 어떻게 계산되는지 보여주는 것입니다

- 그림은 이 과정의 첫 번째 단계를 묘사하는데, 내적 연산을 통해 $x^{(2)}$와 다른 모든 입력 요소 간의 어텐션 점수 ω를 계산하는 것입니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/08.webp" width="400px">

- 입력 시퀀스 요소 2인 $x^{(2)}$를 예시로 사용하여 컨텍스트 벡터 $z^{(2)}$를 계산합니다; 이 섹션의 뒷부분에서 이를 일반화하여 모든 컨텍스트 벡터를 계산할 것입니다.
- 첫 번째 단계는 쿼리 $x^{(2)}$와 다른 모든 입력 토큰 간의 내적을 계산하여 정규화되지 않은 어텐션 점수를 계산하는 것입니다:

In [ ]:
query = inputs[1]  # 2번째 입력 토큰이 쿼리입니다

attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query) # 내적 (1차원 벡터이므로 전치가 필요하지 않음)

print(attn_scores_2)

- 참고: 내적은 본질적으로 두 벡터를 요소별로 곱하고 결과 곱을 합하는 것의 줄임말입니다:

In [ ]:
res = 0.

for idx, element in enumerate(inputs[0]):
    res += inputs[0][idx] * query[idx]

print(res)
print(torch.dot(inputs[0], query))

- **2단계:** 정규화되지 않은 어텐션 점수("오메가", $\omega$)를 합이 1이 되도록 정규화합니다
- 다음은 정규화되지 않은 어텐션 점수를 합이 1이 되도록 정규화하는 간단한 방법입니다 (해석에 유용하고 훈련 안정성에 중요한 관례):

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/09.webp" width="500px">

In [ ]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()

print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

- 하지만 실제로는 극값을 더 잘 처리하고 훈련 중에 더 바람직한 그래디언트 특성을 갖는 소프트맥스 함수를 정규화에 사용하는 것이 일반적이고 권장됩니다.
- 다음은 벡터 요소의 합이 1이 되도록 정규화하는 소프트맥스 함수의 순수한 구현입니다:

In [ ]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)

print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

- 위의 순수한 구현은 오버플로우와 언더플로우 문제로 인해 큰 값이나 작은 입력값에 대해 수치적 불안정성 문제를 겪을 수 있습니다
- 따라서 실제로는 성능에 고도로 최적화된 PyTorch의 소프트맥스 구현을 사용하는 것이 권장됩니다:

In [ ]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)

print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

- **3단계**: 임베딩된 입력 토큰 $x^{(i)}$에 어텐션 가중치를 곱하고 결과 벡터를 합하여 컨텍스트 벡터 $z^{(2)}$를 계산합니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/10.webp" width="500px">

In [ ]:
query = inputs[1] # 2번째 입력 토큰이 쿼리입니다

context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i

print(context_vec_2)

### 3.3.2 모든 입력 토큰에 대한 어텐션 가중치 계산

#### 모든 입력 시퀀스 토큰으로 일반화:

- 위에서는 입력 2에 대한 어텐션 가중치와 컨텍스트 벡터를 계산했습니다 (아래 그림의 강조된 행에 표시된 대로)
- 다음으로, 이 계산을 일반화하여 모든 어텐션 가중치와 컨텍스트 벡터를 계산합니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/11.webp" width="400px">

- (이 그림의 숫자들은 시각적 혼란을 줄이기 위해 소수점 이하 두 자리로 잘렸습니다; 각 행의 값들은 1.0 또는 100%가 되어야 합니다; 마찬가지로 다른 그림의 숫자들도 잘렸습니다)

- 셀프 어텐션에서는 어텐션 점수 계산으로 시작하여, 이를 정규화하여 총합이 1인 어텐션 가중치를 얻습니다
- 그런 다음 이러한 어텐션 가중치를 사용하여 입력의 가중합을 통해 컨텍스트 벡터를 생성합니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/12.webp" width="400px">

- 모든 쌍별 요소에 이전 **1단계**를 적용하여 정규화되지 않은 어텐션 점수 행렬을 계산합니다:

In [ ]:
attn_scores = torch.empty(6, 6)

for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)

print(attn_scores)

- 행렬 곱셈을 통해 위와 동일한 결과를 더 효율적으로 달성할 수 있습니다:

In [ ]:
attn_scores = inputs @ inputs.T
print(attn_scores)

- 이전 **2단계**와 유사하게, 각 행의 값이 1로 합해지도록 각 행을 정규화합니다:

In [ ]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

- 각 행의 값들이 실제로 1로 합해지는지 빠른 검증:

In [ ]:
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("Row 2 sum:", row_2_sum)

print("All row sums:", attn_weights.sum(dim=-1))

- 이전 **3단계**를 적용하여 모든 컨텍스트 벡터를 계산합니다:

In [ ]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

- 이전에 계산한 컨텍스트 벡터 $z^{(2)} = [0.4419, 0.6515, 0.5683]$을 위의 2번째 행에서 찾을 수 있습니다:

In [ ]:
print("Previous 2nd context vector:", context_vec_2)

## 3.4 훈련 가능한 가중치를 가진 셀프 어텐션 구현

- 이 섹션에서 개발된 셀프 어텐션 메커니즘이 이 책과 장의 전체적인 서사 및 구조에 어떻게 통합되는지를 보여주는 개념적 프레임워크

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/13.webp" width="400px">

### 3.4.1 어텐션 가중치를 단계별로 계산하기

- 이 섹션에서는 원본 트랜스포머 아키텍처, GPT 모델, 그리고 대부분의 다른 인기 있는 LLM에서 사용되는 셀프 어텐션 메커니즘을 구현합니다
- 이 셀프 어텐션 메커니즘은 "스케일드 내적 어텐션(scaled dot-product attention)"이라고도 불립니다
- 전반적인 아이디어는 이전과 유사합니다:
  - 특정 입력 요소에 대해 입력 벡터들의 가중합으로 컨텍스트 벡터를 계산하고자 합니다
  - 위를 위해 어텐션 가중치가 필요합니다
- 보시다시피, 앞서 소개한 기본 어텐션 메커니즘과 비교해 약간의 차이점만 있습니다:
  - 가장 주목할 만한 차이점은 모델 훈련 중에 업데이트되는 가중치 행렬의 도입입니다
  - 이러한 훈련 가능한 가중치 행렬은 모델(특히 모델 내의 어텐션 모듈)이 "좋은" 컨텍스트 벡터를 생성하는 것을 학습할 수 있도록 하는 데 중요합니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/14.webp" width="600px">

- 셀프 어텐션 메커니즘을 단계별로 구현하면서, 세 개의 훈련 가중치 행렬 $W_q$, $W_k$, $W_v$를 먼저 소개하겠습니다
- 이 세 행렬은 행렬 곱셈을 통해 임베딩된 입력 토큰 $x^{(i)}$를 쿼리, 키, 값 벡터로 투영하는 데 사용됩니다:

  - 쿼리 벡터: $q^{(i)} = x^{(i)}\,W_q $
  - 키 벡터: $k^{(i)} = x^{(i)}\,W_k $
  - 값 벡터: $v^{(i)} = x^{(i)}\,W_v $

- 입력 $x$와 쿼리 벡터 $q$의 임베딩 차원은 모델의 설계와 특정 구현에 따라 동일하거나 다를 수 있습니다
- GPT 모델에서는 입력과 출력 차원이 보통 동일하지만, 계산을 더 잘 따라할 수 있도록 설명 목적으로 여기서는 다른 입력과 출력 차원을 선택합니다:

In [ ]:
x_2 = inputs[1] # 두 번째 입력 요소
d_in = inputs.shape[1] # 입력 임베딩 크기, d=3
d_out = 2 # 출력 임베딩 크기, d=2

- 아래에서는 세 개의 가중치 행렬을 초기화합니다; 설명 목적으로 출력의 복잡함을 줄이기 위해 `requires_grad=False`로 설정하지만, 모델 훈련을 위해 가중치 행렬을 사용한다면 모델 훈련 중에 이러한 행렬을 업데이트하기 위해 `requires_grad=True`로 설정할 것입니다

In [ ]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

- 다음으로 쿼리, 키, 값 벡터를 계산합니다:

In [ ]:
query_2 = x_2 @ W_query # _2는 두 번째 입력 요소에 대한 것이기 때문입니다
key_2 = x_2 @ W_key 
value_2 = x_2 @ W_value

print(query_2)

- 아래에서 볼 수 있듯이, 6개의 입력 토큰을 3차원에서 2차원 임베딩 공간으로 성공적으로 투영했습니다:

In [ ]:
keys = inputs @ W_key 
values = inputs @ W_value

print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

- 다음 단계인 **2단계**에서는 쿼리와 각 키 벡터 간의 내적을 계산하여 정규화되지 않은 어텐션 점수를 계산합니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/15.webp" width="600px">

In [ ]:
keys_2 = keys[1] # Python은 인덱스를 0부터 시작합니다
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

- 6개의 입력이 있으므로, 주어진 쿼리 벡터에 대해 6개의 어텐션 점수가 있습니다:

In [ ]:
attn_scores_2 = query_2 @ keys.T # 주어진 쿼리에 대한 모든 어텐션 점수
print(attn_scores_2)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/16.webp" width="600px">

- 다음으로 **3단계**에서는 앞서 사용한 소프트맥스 함수를 사용하여 어텐션 가중치(합이 1이 되는 정규화된 어텐션 점수)를 계산합니다
- 앞서와의 차이점은 이제 어텐션 점수를 임베딩 차원의 제곱근 $\sqrt{d_k}$ (즉, `d_k**0.5`)로 나누어 스케일링한다는 것입니다:

In [ ]:
d_k = keys.shape[1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/17.webp" width="600px">

- **4단계**에서는 이제 입력 쿼리 벡터 2에 대한 컨텍스트 벡터를 계산합니다:

In [ ]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

### 3.4.2 간결한 SelfAttention 클래스 구현

- 모든 것을 종합하여 셀프 어텐션 메커니즘을 다음과 같이 구현할 수 있습니다:

In [ ]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        
        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/18.webp" width="400px">

- 바이어스 단위를 비활성화하면 행렬 곱셈과 동등한 PyTorch의 Linear 레이어를 사용하여 위의 구현을 간소화할 수 있습니다
- 수동으로 `nn.Parameter(torch.rand(...)` 접근 방식을 사용하는 것보다 `nn.Linear`를 사용하는 또 다른 큰 장점은 `nn.Linear`가 더 안정적인 모델 훈련으로 이어지는 선호되는 가중치 초기화 스킴을 가지고 있다는 것입니다

In [ ]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

- `SelfAttention_v1`과 `SelfAttention_v2`가 가중치 행렬의 초기 가중치가 다르기 때문에 다른 출력을 주는 것에 주목하세요

## 3.5 인과적 어텐션으로 미래 단어 숨기기

- 인과적 어텐션에서는 대각선 위의 어텐션 가중치가 마스킹되어, 주어진 입력에 대해 LLM이 어텐션 가중치로 컨텍스트 벡터를 계산할 때 미래 토큰을 활용할 수 없도록 합니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/19.webp" width="400px">

### 3.5.1 인과적 어텐션 마스크 적용

- 이 섹션에서는 이전의 셀프 어텐션 메커니즘을 인과적 셀프 어텐션 메커니즘으로 변환합니다
- 인과적 셀프 어텐션은 시퀀스의 특정 위치에 대한 모델의 예측이 이전 위치의 알려진 출력에만 의존하고 미래 위치에는 의존하지 않도록 보장합니다
- 더 간단히 말하면, 각 다음 단어 예측은 선행하는 단어에만 의존해야 한다는 것을 보장합니다
- 이를 달성하기 위해, 각 주어진 토큰에 대해 미래 토큰들(입력 텍스트에서 현재 토큰 이후에 오는 토큰들)을 마스킹합니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/20.webp" width="600px">

- 인과적 셀프 어텐션을 설명하고 구현하기 위해, 이전 섹션의 어텐션 점수와 가중치를 사용해보겠습니다:

In [ ]:
# 편의상 이전 섹션의 SelfAttention_v2 객체의 쿼리와 키 가중치 행렬을 재사용합니다
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs) 
attn_scores = queries @ keys.T

attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

- 미래 어텐션 가중치를 마스킹하는 가장 간단한 방법은 PyTorch의 tril 함수를 사용하여 주 대각선(대각선 자체 포함) 아래의 요소는 1로, 주 대각선 위의 요소는 0으로 설정된 마스크를 생성하는 것입니다:

In [ ]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

- 그런 다음, 어텐션 가중치에 이 마스크를 곱하여 대각선 위의 어텐션 점수를 0으로 만들 수 있습니다:

In [ ]:
masked_simple = attn_weights*mask_simple
print(masked_simple)

- 하지만 위와 같이 소프트맥스 후에 마스크를 적용하면 소프트맥스가 만든 확률 분포를 방해하게 됩니다
- 소프트맥스는 모든 출력값의 합이 1이 되도록 보장합니다
- 소프트맥스 후에 마스킹하면 출력의 합이 다시 1이 되도록 재정규화해야 하므로 과정이 복잡해지고 의도하지 않은 효과를 가져올 수 있습니다

- 행이 1로 합해지도록 하기 위해, 다음과 같이 어텐션 가중치를 정규화할 수 있습니다:

In [ ]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

- 기술적으로는 이제 인과적 어텐션 메커니즘 코딩이 완료되었지만, 위와 동일한 결과를 달성하는 더 효율적인 접근 방식을 간단히 살펴보겠습니다
- 대각선 위의 어텐션 가중치를 0으로 만들고 결과를 재정규화하는 대신, 소프트맥스 함수에 들어가기 전에 대각선 위의 정규화되지 않은 어텐션 점수를 음의 무한대로 마스킹할 수 있습니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/21.webp" width="450px">

In [ ]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

- 아래에서 볼 수 있듯이, 이제 각 행의 어텐션 가중치가 다시 올바르게 1로 합해집니다:

In [ ]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

### 3.5.2 드롭아웃으로 추가 어텐션 가중치 마스킹

- 또한 훈련 중 과적합을 줄이기 위해 드롭아웃을 적용합니다
- 드롭아웃은 여러 곳에 적용할 수 있습니다:
  - 예를 들어, 어텐션 가중치를 계산한 후;
  - 또는 어텐션 가중치를 값 벡터와 곱한 후
- 여기서는 더 일반적이므로 어텐션 가중치를 계산한 후 드롭아웃 마스크를 적용할 것입니다

- 또한 이 특정 예시에서는 50%의 드롭아웃 비율을 사용하는데, 이는 어텐션 가중치의 절반을 무작위로 마스킹한다는 의미입니다. (나중에 GPT 모델을 훈련할 때는 0.1이나 0.2와 같은 더 낮은 드롭아웃 비율을 사용할 것입니다)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/22.webp" width="400px">

- 0.5(50%)의 드롭아웃 비율을 적용하면, 드롭되지 않은 값들은 1/0.5 = 2의 인수로 그에 따라 스케일링됩니다
- 스케일링은 1 / (1 - `dropout_rate`) 공식으로 계산됩니다

In [ ]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5) # 50% 드롭아웃 비율
example = torch.ones(6, 6) # 1로 이루어진 행렬 생성

print(dropout(example))

In [ ]:
torch.manual_seed(123)
print(dropout(attn_weights))

- 결과적인 드롭아웃 출력은 운영 체제에 따라 다르게 보일 수 있습니다; 이 불일치에 대해 더 자세히 알고 싶으시면 [PyTorch 이슈 트래커의 여기](https://github.com/pytorch/pytorch/issues/121595)를 참조하세요

### 3.5.3 간결한 인과적 셀프 어텐션 클래스 구현

- 이제 인과적 및 드롭아웃 마스크를 포함하여 작동하는 셀프 어텐션 구현을 준비할 수 있습니다
- 한 가지 더 구현할 것은 2장에서 구현한 데이터 로더가 생성하는 배치 출력을 우리의 `CausalAttention` 클래스가 지원할 수 있도록 하나 이상의 입력으로 구성된 배치를 처리하는 코드입니다
- 단순화를 위해, 이러한 배치 입력을 시뮬레이션하기 위해 입력 텍스트 예시를 복제합니다:

In [ ]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape) # 각각 6개의 토큰을 가진 2개의 입력, 각 토큰은 임베딩 차원 3을 가집니다

In [ ]:
class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout) # 새로 추가됨
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) # 새로 추가됨

    def forward(self, x):
        b, num_tokens, d_in = x.shape # 새로운 배치 차원 b
        # `num_tokens`가 `context_length`를 초과하는 입력의 경우, 아래 마스크 생성에서 오류가 발생합니다.
        # 실제로는 LLM(4-7장)이 이 forward 메서드에 도달하기 전에 입력이 
        # `context_length`를 초과하지 않도록 보장하므로 문제가 되지 않습니다.
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2) # 전치 변경됨
        attn_scores.masked_fill_(  # 새로 추가됨, _ 연산은 in-place
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)  # `:num_tokens`는 배치의 토큰 수가 지원되는 context_size보다 작은 경우를 고려
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights) # 새로 추가됨

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)

context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)

context_vecs = ca(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

- 드롭아웃은 추론 시가 아니라 훈련 중에만 적용됩니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/23.webp" width="500px">

## 3.6 단일 헤드 어텐션을 멀티 헤드 어텐션으로 확장

### 3.6.1 여러 단일 헤드 어텐션 레이어 쌓기

- 아래는 이전에 구현한 셀프 어텐션의 요약입니다 (단순화를 위해 인과적 및 드롭아웃 마스크는 표시하지 않음)

- 이는 단일 헤드 어텐션이라고도 불립니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/24.webp" width="400px">

- 멀티 헤드 어텐션 모듈을 얻기 위해 여러 단일 헤드 어텐션 모듈을 단순히 쌓습니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/25.webp" width="400px">

- 멀티 헤드 어텐션의 주요 아이디어는 서로 다른 학습된 선형 투영을 사용하여 어텐션 메커니즘을 여러 번 (병렬로) 실행하는 것입니다. 이를 통해 모델이 서로 다른 위치에서 다른 표현 부공간의 정보에 공동으로 주의를 기울일 수 있습니다.

In [ ]:
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) 
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


torch.manual_seed(123)

context_length = batch.shape[1] # 이는 토큰의 수입니다
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

- 위의 구현에서 임베딩 차원은 4입니다. 키, 쿼리, 값 벡터와 컨텍스트 벡터의 임베딩 차원으로 `d_out=2`를 사용했고, 2개의 어텐션 헤드가 있으므로 출력 임베딩 차원이 2*2=4가 됩니다

### 3.6.2 가중치 분할을 사용한 멀티 헤드 어텐션 구현

- 위는 멀티 헤드 어텐션의 직관적이고 완전히 기능적인 구현입니다 (앞서 구현한 단일 헤드 어텐션 `CausalAttention`을 래핑), `MultiHeadAttention`이라는 독립형 클래스를 작성하여 동일한 결과를 달성할 수 있습니다

- 이 독립형 `MultiHeadAttention` 클래스에서는 단일 어텐션 헤드를 연결하지 않습니다
- 대신, 단일 W_query, W_key, W_value 가중치 행렬을 생성한 후 이를 각 어텐션 헤드에 대한 개별 행렬로 분할합니다:

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # 원하는 출력 차원에 맞추기 위해 투영 차원을 축소

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # 헤드 출력을 결합하는 선형 레이어
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # `CausalAttention`에서와 같이, `num_tokens`가 `context_length`를 초과하는 입력의 경우
        # 아래 마스크 생성에서 오류가 발생합니다.
        # 실제로는 LLM(4-7장)이 이 forward 메서드에 도달하기 전에 입력이 
        # `context_length`를 초과하지 않도록 보장하므로 문제가 되지 않습니다.

        keys = self.W_key(x) # 형태: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # `num_heads` 차원을 추가하여 암시적으로 행렬을 분할합니다
        # 마지막 차원을 펼칩니다: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # 전치: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # 인과적 마스크를 사용한 스케일드 내적 어텐션 계산 (셀프 어텐션)
        attn_scores = queries @ keys.transpose(2, 3)  # 각 헤드에 대한 내적

        # 토큰 수로 잘리고 불린으로 변환된 원래 마스크
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # 마스크를 사용하여 어텐션 점수 채우기
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # 형태: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # 헤드들을 결합, 여기서 self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # 선택적 투영

        return context_vec

torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

- 위는 본질적으로 더 효율적인 `MultiHeadAttentionWrapper`의 재작성된 버전입니다
- 랜덤 가중치 초기화가 다르기 때문에 결과 출력이 약간 다르게 보이지만, 둘 다 다가오는 장에서 구현할 GPT 클래스에서 사용할 수 있는 완전히 기능적인 구현입니다
- 또한, 위의 `MultiHeadAttention` 클래스에 선형 투영 레이어(`self.out_proj`)를 추가했습니다. 이는 차원을 변경하지 않는 단순한 선형 변환입니다. LLM 구현에서 이러한 투영 레이어를 사용하는 것이 표준 관례이지만, 꼭 필요한 것은 아닙니다 (최근 연구에서는 모델링 성능에 영향을 주지 않고 제거할 수 있다고 보여졌습니다; 이 장 끝의 추가 읽기 섹션 참조)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/26.webp" width="400px">

- 위의 간결하고 효율적인 구현에 관심이 있으시다면, PyTorch의 [`torch.nn.MultiheadAttention`](https://pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html) 클래스도 고려해볼 수 있습니다

- 위의 구현이 처음에는 약간 복잡해 보일 수 있으므로, `attn_scores = queries @ keys.transpose(2, 3)`를 실행할 때 무슨 일이 일어나는지 살펴보겠습니다:

In [ ]:
# (b, num_heads, num_tokens, head_dim) = (1, 2, 3, 4)
a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573],
                    [0.8993, 0.0390, 0.9268, 0.7388],
                    [0.7179, 0.7058, 0.9156, 0.4340]],

                   [[0.0772, 0.3565, 0.1479, 0.5331],
                    [0.4066, 0.2318, 0.4545, 0.9737],
                    [0.4606, 0.5159, 0.4220, 0.5786]]]])

print(a @ a.transpose(2, 3))

- 이 경우, PyTorch의 행렬 곱셈 구현은 4차원 입력 텐서를 처리하여 마지막 2개의 차원(num_tokens, head_dim) 간에 행렬 곱셈이 수행되고 개별 헤드에 대해 반복됩니다

- 예를 들어, 다음은 각 헤드에 대해 별도로 행렬 곱셈을 계산하는 더 간결한 방법이 됩니다:

In [ ]:
first_head = a[0, 0, :, :]
first_res = first_head @ first_head.T
print("First head:\n", first_res)

second_head = a[0, 1, :, :]
second_res = second_head @ second_head.T
print("\nSecond head:\n", second_res)

# 요약 및 핵심사항

- 데이터 로더(2장)와 이 장에서 구현한 멀티 헤드 어텐션 클래스의 간결한 버전인 [./multihead-attention.ipynb](./multihead-attention.ipynb) 코드 노트북을 참조하세요. 이는 다가오는 장에서 GPT 모델을 훈련하는 데 필요합니다
- 연습 문제 해답은 [./exercise-solutions.ipynb](./exercise-solutions.ipynb)에서 찾을 수 있습니다